# Sandbox: Get familiar with the thermal set-up

![alt text](start_measuring_dspace.png "Start Measurement Procedure")

 Press "Start Measuring" in D-Space Before continuing!

In [ ]:
# Dependencies and imports - RUN THIS CELL before all others
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import ipywidgets as widgets
import csv
import os
from datetime import datetime
import time
import threading
from DS1104_interface_V2 import ThermalControlApparatus, CollectMeasurementData, AnimateScope

### The code below generates a scope, which reads the input- and output signals from d-space. Run the cell to turn on the scope

In [ ]:
# Signal Scope Cell
ani = None # Global variable to hold the animation object
plt.close("all") # Close all previous figures to avoid overlap

plant = ThermalControlApparatus() # Load thermal control apparatus interface
plant.start() # Start the apparatus (connects and starts safty thread)   
window_length = 20 # seconds
dt = 0.1 # seconds
def run_T_scope():
    global ani
    if ani is not None:
        ani.event_source.stop()  # Stop the existing animation

    plt.close("all")  # Close all previous figures to avoid overlap
    # Create two subplots: one for temperatures, one for inputs
    fig, (ax_temp, ax_input) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

    # Temperature lines
    line1, = ax_temp.plot(np.linspace(0, window_length, int(window_length / dt)), np.zeros(int(window_length / dt)), label='T_1')
    line2, = ax_temp.plot(np.linspace(0, window_length, int(window_length / dt)), np.zeros(int(window_length / dt)), label='T_2')

    # Input lines
    line3, = ax_input.plot(np.linspace(0, window_length, int(window_length / dt)), np.zeros(int(window_length / dt)), label='Heater')
    line4, = ax_input.plot(np.linspace(0, window_length, int(window_length / dt)), np.zeros(int(window_length / dt)), label='Fan')
    line5, = ax_input.plot(np.linspace(0, window_length, int(window_length / dt)), np.zeros(int(window_length / dt)), label='Vane')
    
    # Plot settings, feel free to change these how you see fit
    #Temperature plot settings
    ax_temp.legend()
    ax_temp.set_ylim(0, 100)
    ax_temp.set_title("Live Temperature Readings")
    ax_temp.set_ylabel("Temperature (°C)")
    ax_temp.grid(True)

    #Input signal plot settings
    ax_input.legend()
    ax_input.set_ylim(0, 12)
    ax_input.set_title("Supplied Inputs")
    ax_input.set_xlabel("Time (s)")
    ax_input.set_ylabel("Input Value (V)")
    ax_input.grid(True)

    # Use the AnimateScope signature
    state = AnimateScope([line1, line2], [line3, line4, line5], ax_temp, ax_input, window_length=window_length, dt=dt, plant=plant, plant_model=None)
    ani = animation.FuncAnimation(fig,
                                 state.animate,
                                 init_func=state.init_plot,
                                 interval=100,
                                 blit=True,
                                 save_count=0)

    plt.tight_layout()
    plt.show()

run_T_scope()

### Run the code below to change the inputs. This can be done with the sliders

In [ ]:
HeatSlider = widgets.FloatSlider(value = 0.0, min = 0.0, max = 10.0, step = 0.01, description='Heater Power')
FanSlider = widgets.FloatSlider(value = 0.0, min = 0.0, max = 10.0, step = 0.01, description='Fan Power')
VaneSlider = widgets.FloatSlider(value = 0.0, min = 0.0, max = 10.0, step = 0.01, description='Vane Rotation')

widgets.interact(plant.write_heater, heater_input=HeatSlider);
widgets.interact(plant.write_fan, fan_input=FanSlider);
widgets.interact(plant.write_vane, vane_input=VaneSlider);

### After you're done experimenting, always run the cool-down command cell before exiting the environment

In [ ]:
# cool down the system when done experimenting
plant.cooling_down()